# 02 · Our Evaluation Script: Sanity Check on a Kaggle T4

**Project:** Silent Tool-Call Failures: How Quantization Degrades Function Calling in Small LLMs
**Update 3 goal:** run our own evaluation script (`src/run_eval.py` and `src/score.py`) on the model from Update 2, and check that its score is close to the BFCL run on the same GPU. Every later run uses this script.

**Model:** `Qwen/Qwen3-4B-Instruct-2507` in FP16. In Update 2, BFCL's own harness scored it 86.31% (prompt mode) and 87.81% (FC mode) on a T4.
Our script asks for a JSON array of calls instead of BFCL's Python call syntax, so a few points apart is expected. A large gap would point to a bug in the prompt or the parser.

**Runs:**
1. **Free mode** on all 1,390 `non_live` questions: the sanity check.
2. **Constrained mode** (JSON-schema-guided decoding) on the first 3 questions of each category (21 questions): only a check that constrained decoding works on the T4. The full constrained runs come in Week 5.

## How to run on Kaggle

1. kaggle.com → **Create → New Notebook → File → Import Notebook**, then upload this file.
2. In the right panel, under **Settings**: **Accelerator → GPU T4 x2** and **Internet → On**.
3. Not needed for this model: an `HF_TOKEN` secret (**Add-ons → Secrets**). Llama 3.2 will need it later.
4. Click **Save Version → Save & Run All (Commit)**. It runs in the background, so you can close the tab. To watch it live instead, use **Run All**.

**Time:** about 20-30 minutes in total, including setup.
**Output:** `update3_results.zip` with the raw answers, scores, run information and `eval_sanity_check.md`.

Each step checks its own result and stops with a clear message if something is wrong.

## 1. Settings
Edit this cell only.

In [2]:
import os

REPO_URL = "https://github.com/shawalkabirchy/quantization-functional-call"
REPO_REF = "main"             # branch, tag or commit of the code to run
CONFIG = "configs/qwen3-4b-instruct-2507_fp16.yaml"
SMOKE_TEST_LIMIT = 3          # constrained mode: first 3 questions of each category (21 in total)

BFCL_VERSION = "2026.3.23"    # pinned for reproducibility
WORK = "/kaggle/working"
REPO = f"{WORK}/repo"                         # code and results (kept as notebook output)
VENV = os.path.expanduser("~/bfcl-venv")      # separate Python 3.11 environment (not kept)

CONFIG_NAME = os.path.splitext(os.path.basename(CONFIG))[0]
RUN_FREE = f"{CONFIG_NAME}_free"
RUN_CONSTRAINED = f"{CONFIG_NAME}_constrained_limit{SMOKE_TEST_LIMIT}"

## 2. Check the GPU and internet

In [3]:
import json
import shutil
import subprocess
from datetime import date

import requests

# GPU
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if smi.returncode != 0:
    raise RuntimeError("No GPU found. Set Settings -> Accelerator -> GPU T4 x2, then run again.")
print("GPUs:\n" + smi.stdout)

# Internet
try:
    requests.get("https://pypi.org/simple/", timeout=15).raise_for_status()
    print("Internet: OK")
except Exception as e:
    raise RuntimeError("No internet. Set Settings -> Internet -> On (needs a phone-verified Kaggle account).") from e

# Hugging Face token: optional for Qwen, required for gated models such as Llama 3.2
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: loaded from Kaggle Secrets")
except Exception:
    print("HF_TOKEN: not set. Fine for Qwen (not gated); add it under Add-ons -> Secrets before running Llama.")

GPUs:
0, Tesla T4, 15360 MiB, 580.178.04
1, Tesla T4, 15360 MiB, 580.178.04

Internet: OK
HF_TOKEN: not set. Fine for Qwen (not gated); add it under Add-ons -> Secrets before running Llama.


## 3. Install BFCL and vLLM (about 1-5 minutes)
The same environment as in notebook 01: a separate Python 3.11 environment with `bfcl-eval[oss_eval_vllm]` (vLLM 0.8.5) and the same pins. See notebook 01 for why each pin is there.

In [4]:
!pip install -q uv
shutil.rmtree(VENV, ignore_errors=True)
!uv venv {VENV} --python 3.11
!uv pip install --python {VENV}/bin/python "bfcl-eval[oss_eval_vllm]=={BFCL_VERSION}" "transformers==4.51.3" "openai>=1.86.0,<2" "fastapi<0.116" "pydantic<2.12" "setuptools>=77.0.3,<80" soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 78.2 MB/s eta 0:00:00:00:0100:01
Using CPython 3.11.17
Creating virtual environment at: /root/bfcl-venv
Activate with: source /root/bfcl-venv/bin/activate
Using Python 3.11.17 environment at: /root/bfcl-venv
Resolved 213 packages in 1.97s                                       
Prepared 213 packages in 56.60s                                          
Installed 213 packages in 530ms                             
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + airportsdata==20260905
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==1.13.0
 + anyio==4.15.1
 + argcomplete==3.7.2
 + astor==0.8.1
 + attrs==26.1.0
 + beautifulsoup4==4.15.0
 + bfcl-eval==2026.3.23
 + black==26.10.1
 + blake3==1.0.11
 + boto3==1.43.111
 + botocore==1.43.111
 + cachetools==7.2.1
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + cohere==5.18.0


## 4. Get the code and check the installation
Clones the repository at `REPO_REF`. The commit is saved with every run (`results/meta/`), so each result can be traced to the exact code that made it.

In [5]:
shutil.rmtree(REPO, ignore_errors=True)
!git clone --quiet {REPO_URL} {REPO}
!git -C {REPO} checkout --quiet {REPO_REF}
!git -C {REPO} log -1 --format="Code version: %h %s (%ci)"

CHECK = (
    "import vllm, bfcl_eval, yaml, xgrammar, triton, torch; from importlib.metadata import version; "
    "print('vllm', version('vllm'), '| xgrammar', version('xgrammar'), '| torch', version('torch')); "
    "print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"
)
r = subprocess.run([f"{VENV}/bin/python", "-c", CHECK], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-5000:])
    raise RuntimeError("Environment check failed (see above). Check the output of step 3.")

freeze = subprocess.run(["uv", "pip", "freeze", "--python", f"{VENV}/bin/python"], capture_output=True, text=True)
with open(f"{WORK}/requirements-freeze.txt", "w") as f:
    f.write(freeze.stdout)
print(f"Saved {len(freeze.stdout.splitlines())} package versions to {WORK}/requirements-freeze.txt")

Code version: e5b2d67 Update 3: own evaluation script, scorer, configs for all models (2026-10-12 13:16:09 +0600)
INFO 10-12 07:55:46 [__init__.py:239] Automatically detected platform cuda.
vllm 0.8.5 | xgrammar 0.1.18 | torch 2.6.0
GPUs: ['Tesla T4', 'Tesla T4']

Saved 213 package versions to /kaggle/working/requirements-freeze.txt


## 5. Free mode on all 1,390 questions (about 10 minutes)
Greedy decoding (temperature 0) with every decoding setting taken from the config file. Raw answers go to `results/raw/`, run information (versions, GPU, memory, speed) to `results/meta/`.

In [6]:
!cd {REPO} && {VENV}/bin/python src/run_eval.py --config {CONFIG} --mode free --overwrite 2>&1 | tee {WORK}/run_free.log

if not os.path.exists(f"{REPO}/results/meta/{RUN_FREE}.json"):
    raise RuntimeError(f"run_eval.py did not finish (see the output above, also saved in {WORK}/run_free.log).")

Run qwen3-4b-instruct-2507_fp16_free: 1390 questions from simple_python, simple_java, simple_javascript, multiple, parallel, parallel_multiple, irrelevance
INFO 10-12 08:17:13 [__init__.py:239] Automatically detected platform cuda.
WARNING 10-12 08:17:16 [config.py:2972] Casting torch.bfloat16 to torch.float16.
INFO 10-12 08:17:27 [config.py:717] This model supports multiple tasks: {'generate', 'reward', 'classify', 'score', 'embed'}. Defaulting to 'generate'.
INFO 10-12 08:17:27 [llm_engine.py:240] Initializing a V0 LLM engine (v0.8.5) with config: model='Qwen/Qwen3-4B-Instruct-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Instruct-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=cdbee75f17c01a7cc42f958dc650907174af0554, override_neuron_config=None, tokenizer_revision=cdbee75f17c01a7cc42f958dc650907174af0554, trust_remote_code=False, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_paralle

## 6. Score the answers
`src/score.py` parses each answer and checks it with BFCL's own AST checker. (Before this run, it was checked on the Update 2 answers: it gives exactly BFCL's scores on all 1,390 questions.)

In [7]:
!cd {REPO} && {VENV}/bin/python src/score.py results/raw/{RUN_FREE}.jsonl

if not os.path.exists(f"{REPO}/results/scores/{RUN_FREE}.jsonl"):
    raise RuntimeError("score.py did not finish (see the output above).")


qwen3-4b-instruct-2507_fp16_free  (1390 questions)
  category               correct  accuracy   parsed
  simple_python          384/400     96.00   100.00
  simple_java             65/100     65.00    96.00
  simple_javascript        36/50     72.00    90.00
  multiple               188/200     94.00   100.00
  parallel               188/200     94.00   100.00
  parallel_multiple      183/200     91.50   100.00
  irrelevance            206/240     85.83   100.00
  simple_ast                         77.67         
  non_live_overall                   89.29         
  saved /kaggle/working/repo/results/scores/qwen3-4b-instruct-2507_fp16_free.jsonl and /kaggle/working/repo/results/summary.csv


## 7. Compare with the BFCL run from Update 2
The check passes if the overall score is within 5 points of at least one of the two BFCL runs (prompt or FC mode) on the same model and GPU. "Parsed" is the share of answers that were valid JSON in the requested format; if it is low, look at the examples printed below before anything else.

In [8]:
import pandas as pd

TOLERANCE = 5.0  # percentage points, on Non-Live Overall Acc
NAMES = {
    "non_live_overall": "Non-Live Overall Acc",
    "simple_python": "Python Simple AST",
    "simple_java": "Java Simple AST",
    "simple_javascript": "JavaScript Simple AST",
    "multiple": "Multiple AST",
    "parallel": "Parallel AST",
    "parallel_multiple": "Parallel Multiple AST",
    "irrelevance": "Irrelevance Detection",
}

summary = pd.read_csv(f"{REPO}/results/summary.csv")
ours = summary[summary["run_id"] == RUN_FREE].set_index("category")
bfcl = pd.read_csv(f"{REPO}/results/bfcl_repro/comparison.csv")


def bfcl_score(mode, column):
    row = bfcl[(bfcl["Model"] == f"Qwen3-4B-Instruct-2507 ({mode})") & (bfcl["Category"] == column)]
    return float(row["Ours (FP16, T4)"].iloc[0])


rows = []
for category, column in NAMES.items():
    o = float(ours.loc[category, "accuracy"])
    p, fc = bfcl_score("Prompt", column), bfcl_score("FC", column)
    parsed = ours.loc[category, "parsed_pct"]
    rows.append({
        "Category": column,
        "Our script": o,
        "BFCL prompt": p,
        "BFCL FC": fc,
        "vs prompt": round(o - p, 2),
        "vs FC": round(o - fc, 2),
        "Parsed (%)": None if pd.isna(parsed) else float(parsed),
    })
comparison = pd.DataFrame(rows)
comparison.to_csv(f"{WORK}/eval_sanity_comparison.csv", index=False)
display(comparison)

overall = comparison.iloc[0]
gap = min(abs(overall["vs prompt"]), abs(overall["vs FC"]))
if gap <= TOLERANCE:
    verdict = (f"Pass: the overall score ({overall['Our script']:.2f}%) is {gap:.2f} points from the closer "
               f"BFCL run (tolerance {TOLERANCE:.0f} points).")
else:
    verdict = (f"Check needed: the overall score ({overall['Our script']:.2f}%) is {gap:.2f} points from the "
               f"closer BFCL run (tolerance {TOLERANCE:.0f} points). Look at the parse failures below first.")
print(verdict)

# What went wrong, and a few examples of each kind
scores = pd.read_json(f"{REPO}/results/scores/{RUN_FREE}.jsonl", lines=True)
raw = pd.read_json(f"{REPO}/results/raw/{RUN_FREE}.jsonl", lines=True).set_index("id")
with open(f"{REPO}/results/meta/{RUN_FREE}.json") as f:
    meta = json.load(f)
failures = scores[~scores["valid"]]
print("\nFailures by type:")
print(failures.groupby(["category", "error_type"]).size().to_string())
for title, subset in [("Answers that did not parse", failures[~failures["parse_ok"]]),
                      ("Answers that parsed but were wrong", failures[failures["parse_ok"]])]:
    print(f"\n{title}: {len(subset)}")
    for _, s in subset.head(4).iterrows():
        print(f"  {s['id']}  ({s['error_type']})")
        print("    output:", repr(raw.loc[s["id"], "output"][:300]))
        print("    error: ", str(s["error"])[:300])
print("\nFinish reasons:", meta["finish_reasons"])

,Category,Our script,BFCL prompt,BFCL FC,vs prompt,vs FC,Parsed (%)
0,Non-Live Overall Acc,89.29,86.31,87.81,2.98,1.48,NaN
1,Python Simple AST,96.00,93.75,95.25,2.25,0.75,100.0
2,Java Simple AST,65.00,64.00,63.00,1.00,2.00,96.0
3,JavaScript Simple AST,72.00,74.00,66.00,-2.00,6.00,90.0
4,Multiple AST,94.00,91.00,94.00,3.00,0.00,100.0
5,Parallel AST,94.00,88.00,92.50,6.00,1.50,100.0
6,Parallel Multiple AST,91.50,89.00,90.00,2.50,1.50,100.0
7,Irrelevance Detection,85.83,84.58,88.75,1.25,-2.92,100.0


Pass: the overall score (89.29%) is 1.48 points from the closer BFCL run (tolerance 5 points).

Failures by type:
category           error_type                                          
irrelevance        irrelevance_error:decoder_success                       34
multiple           multiple_function_checker:wrong_count                    4
                   simple_function_checker:missing_optional                 2
                   simple_function_checker:wrong_func_name                  1
                   value_error:others                                       2
                   value_error:string                                       3
parallel           parallel_function_checker_no_order:cannot_find_match     3
                   parallel_function_checker_no_order:wrong_count           9
parallel_multiple  parallel_function_checker_no_order:cannot_find_match    14
                   parallel_function_checker_no_order:wrong_count           3
simple_java        parse_error    

## 8. Constrained mode: quick check on 21 questions
Every answer should parse (Parsed = 100%), because the output is forced to match the JSON schema. Scores on 3 questions per category say nothing about accuracy yet.

In [9]:
# If this check fails, the notebook goes on: the free-mode results above are still reported and packaged.
!cd {REPO} && {VENV}/bin/python src/run_eval.py --config {CONFIG} --mode constrained --limit {SMOKE_TEST_LIMIT} --overwrite 2>&1 | tee {WORK}/run_constrained.log

parsed_all = None  # stays None if the constrained run did not finish
if os.path.exists(f"{REPO}/results/meta/{RUN_CONSTRAINED}.json"):
    !cd {REPO} && {VENV}/bin/python src/score.py results/raw/{RUN_CONSTRAINED}.jsonl
    c_scores = pd.read_json(f"{REPO}/results/scores/{RUN_CONSTRAINED}.jsonl", lines=True)
    c_raw = pd.read_json(f"{REPO}/results/raw/{RUN_CONSTRAINED}.jsonl", lines=True).set_index("id")
    for _, s in c_scores.head(8).iterrows():
        print(f"{s['id']:22s} parsed={s['parse_ok']!s:5s} correct={s['valid']!s:5s} {c_raw.loc[s['id'], 'output'][:150]!r}")
    parsed_all = bool(c_scores["parse_ok"].all())
    print("\nConstrained decoding:", "every answer parsed." if parsed_all else
          "SOME ANSWERS DID NOT PARSE. Check the outputs above (for example, answers cut off at max_tokens).")
else:
    print(f"CONSTRAINED RUN DID NOT FINISH. See the output above (also in {WORK}/run_constrained.log). "
          "The free-mode results are still packaged in the next step.")

Run qwen3-4b-instruct-2507_fp16_constrained_limit3: 21 questions from simple_python, simple_java, simple_javascript, multiple, parallel, parallel_multiple, irrelevance
INFO 10-12 08:49:13 [__init__.py:239] Automatically detected platform cuda.
WARNING 10-12 08:49:16 [config.py:2972] Casting torch.bfloat16 to torch.float16.
INFO 10-12 08:49:26 [config.py:717] This model supports multiple tasks: {'classify', 'embed', 'score', 'reward', 'generate'}. Defaulting to 'generate'.
INFO 10-12 08:49:26 [llm_engine.py:240] Initializing a V0 LLM engine (v0.8.5) with config: model='Qwen/Qwen3-4B-Instruct-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Instruct-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=cdbee75f17c01a7cc42f958dc650907174af0554, override_neuron_config=None, tokenizer_revision=cdbee75f17c01a7cc42f958dc650907174af0554, trust_remote_code=False, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipe

## 9. Write the report and package the results

In [10]:
c_meta = None
if parsed_all is not None:
    with open(f"{REPO}/results/meta/{RUN_CONSTRAINED}.json") as f:
        c_meta = json.load(f)
v, mem = meta["versions"], meta["memory"]
fmt = lambda x: "n/a" if x is None or pd.isna(x) else f"{x:.2f}"
lines = [
    "# Evaluation Script Sanity Check (Update 3)",
    "",
    f"**Date:** {date.today()}",
    "**Goal:** check that our own evaluation script gives scores close to BFCL's own harness, on the same model and GPU.",
    "",
    "## Setup",
    "",
    f"- **Model:** `{meta['model']}` (revision `{meta['revision'][:12]}`), {meta['precision'].upper()}",
    f"- **Questions:** BFCL `non_live`, {meta['n_questions']} questions, bfcl-eval {v['bfcl-eval']}",
    f"- **Prompt format:** `{meta['prompt_format']}`: one system prompt asking for a JSON array of calls",
    f"- **Decoding:** `{meta['sampling_params']}`",
    f"- **Inference:** vLLM {v['vllm']} on {', '.join(meta['gpus'])}; tensor parallel size {meta['config']['tensor_parallel_size']}",
    f"- **Software:** Python {v['python']}, torch {v['torch']}, transformers {v['transformers']}, xgrammar {v['xgrammar']}",
    f"- **Code:** commit `{(meta['code']['commit'] or 'unknown')[:7]}`",
    "",
    "## Results: free mode (accuracy, %)",
    "",
    "| Category | Our script | BFCL prompt | BFCL FC | vs prompt | vs FC | Parsed (%) |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for r in rows:
    lines.append(f"| {r['Category']} | {fmt(r['Our script'])} | {fmt(r['BFCL prompt'])} | {fmt(r['BFCL FC'])} | "
                 f"{fmt(r['vs prompt'])} | {fmt(r['vs FC'])} | {fmt(r['Parsed (%)'])} |")
lines += [
    "",
    f"**Verdict:** {verdict}",
    "",
    "## Run information",
    "",
    f"- Generation: {meta['generation_seconds']:.0f} s for {meta['n_questions']} questions, "
    f"{meta['output_tokens_per_second']} output tokens/s; model loading {meta['model_load_seconds']:.0f} s",
    f"- Finish reasons: {meta['finish_reasons']}",
    f"- Memory (vLLM): weights {mem['model_weights_gib']} GiB, activation peak {mem['activation_peak_gib']} GiB, "
    f"KV cache {mem['kv_cache_gib']} GiB",
    ("- Constrained mode: the run did not finish (see run_constrained.log)" if c_meta is None else
     f"- Constrained mode, {c_meta['n_questions']} questions: "
     + ("every answer parsed" if parsed_all else "NOT every answer parsed")
     + f"; {c_meta['generation_seconds']:.0f} s"),
    "",
    "## Notes",
    "",
    "- ",
]
with open(f"{WORK}/eval_sanity_check.md", "w") as f:
    f.write("\n".join(lines) + "\n")
print("\n".join(lines))

# Package: results/ (raw answers, run information, scores, summary) plus the report and logs
pack = f"{WORK}/update3_results"
shutil.rmtree(pack, ignore_errors=True)
shutil.copytree(f"{REPO}/results", f"{pack}/results",
                ignore=shutil.ignore_patterns("bfcl_repro", "figures"))
for name in ["eval_sanity_check.md", "eval_sanity_comparison.csv", "requirements-freeze.txt",
             "run_free.log", "run_constrained.log"]:
    if os.path.exists(f"{WORK}/{name}"):
        shutil.copy(f"{WORK}/{name}", pack)
archive = shutil.make_archive(pack, "zip", pack)
print("\nSaved", archive)

# Evaluation Script Sanity Check (Update 3)

**Date:** 2026-10-12
**Goal:** check that our own evaluation script gives scores close to BFCL's own harness, on the same model and GPU.

## Setup

- **Model:** `Qwen/Qwen3-4B-Instruct-2507` (revision `cdbee75f17c0`), FP16
- **Questions:** BFCL `non_live`, 1390 questions, bfcl-eval 2026.3.23
- **Prompt format:** `json-array-v1`: one system prompt asking for a JSON array of calls
- **Decoding:** `SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.0, top_p=1.0, top_k=-1, min_p=0.0, seed=42, stop=[], stop_token_ids=[], bad_words=[], include_stop_str_in_output=False, ignore_eos=False, max_tokens=4096, min_tokens=0, logprobs=None, prompt_logprobs=None, skip_special_tokens=True, spaces_between_special_tokens=True, truncate_prompt_tokens=None, guided_decoding=None, extra_args=None)`
- **Inference:** vLLM 0.8.5 on Tesla T4, Tesla T4; tensor parallel size 1
- **Software:** Python 3.11.17, torch 2.6.

## After the run: what goes into the repository

1. Download **`update3_results.zip`** from the notebook's **Output** section (or from the saved version's Output tab).
2. Download the notebook with its outputs (**File → Download Notebook**) and replace `notebooks/02_eval_sanity_check.ipynb` with it.
3. From the zip:
   - `eval_sanity_check.md` → `notes/eval_sanity_check.md`; write a sentence or two under **Notes**
   - `results/meta/`, `results/scores/` and `results/summary.csv` → the same places in the repository
   - `requirements-freeze.txt` → `results/meta/requirements-freeze_update3.txt`
   - The raw answers in `results/raw/` stay out of git (`results/raw/` is ignored); keep the zip as a backup.
4. Add the GPU time to `notes/compute_log.md`, add the Update 3 entry at the top of `UPDATES.md`, then commit and push before the meeting.